# FMG-LightNet: Federated Multimodal Graph Lightweight Network
**Patient-specific coronary plaque progression forecasting and cardiovascular risk stratification**

Implements, end to end: IBR-CEI preprocessing, L3DU-MAT segmentation, **MP-CTNet**, **RQ-UOTFNet**, **EC-SGAT**, **TQM-FedNet** (FiLM hypernetwork, tree-quality weighted aggregation, local DP-SGD, RDP accountant), **PPS-GODENet** (dissipative graph ODE + monotone Breslow-Cox head), evaluation, ablations, theory checks and edge benchmarking.

**Data modes**
* `real`: put the datasets under `CFG['data_root']` (layout in the data cell). NHANES `.xpt` + mortality `.dat` and ImageCAS-X NIfTI volumes are loaded.
* `synthetic` (automatic fallback): a physics-consistent generator of canonical coronary trees and an NHANES-like survival cohort, so the whole pipeline runs and can be debugged offline.

**Honest notes** (also repeated where relevant)
1. ImageCAS-X has no survival follow-up and NHANES has no imaging. Survival labels for imaging patients are therefore taken from an **atlas-matched surrogate NHANES subject** (kernel match on severity rank, within the same split). This is a modelling assumption of the unpaired setting, not ground truth.
2. Public CCTA data has no serial scans. Real-data progression targets need a longitudinal cohort; the synthetic mode uses a natural-history simulator.
3. The numbers in the paper (C-index 0.864, 62.4 FPS, 38,905 parameters) are not reproduced by construction; the notebook prints what *this* implementation obtains.

In [ ]:
import os, math, json, time, copy, glob, warnings, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
warnings.filterwarnings("ignore")

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

CFG = dict(
    seed=42, data_root="./data", mode="auto",          # auto | real | synthetic
    # cohort sizes (synthetic mode)
    n_img=600, n_nhanes=3000,
    # graph / model
    n_slots=18, max_st=8, d_st=9, hidden=32, heads=2, conv_layers=2, gat_layers=2,
    # RQ-UOTFNet
    n_bins=16, min_bin=20, uot_eps=0.05, uot_ta=1.0, uot_tb=1.0, sens=2.0, sink_iters=30, match_bw=0.05,
    # TQM-FedNet
    n_clients=5, dirichlet=0.5, rounds=10, local_epochs=1, batch=16, lr=2e-3, wd=1e-4,
    mu_prox=0.01, clip=1.0, sigma=1.0, delta=1e-5, align_beta=2.0, film_bound=0.5, site_dim=4,
    # PPS-GODENet
    horizon_years=5.0, ode_steps=8, n_proto=8, proto_dim=16, mono_units=8, proto_temp=1.0, d_floor=0.2,
    # losses
    w_prog=1.0, w_surv=1.0, w_rank=0.5, w_proto=0.05, w_phys=0.1, w_nr=0.5, w_stage=0.1,
    rank_margin=0.1, rank_temp=0.5,
    # segmentation preprocessing
    voxel_mm=0.5, hu_lo=-100.0, hu_hi=700.0,
    ablation_rounds=5,
)
QUICK = False      # True: small smoke-test run (~minutes on CPU)
if QUICK: CFG.update(n_img=240, n_nhanes=1500, rounds=3, ablation_rounds=2, n_bins=8, min_bin=15)
S = CFG["n_slots"]; NST = S * CFG["max_st"]
seed_all(CFG["seed"])

# ---- canonical 18-slot tree (slot 0 = virtual ostial root) ----
SLOT_NAMES = ["root","RCA-p","RCA-m","RCA-d","PDA","R-PLB","LM","LAD-p","LAD-m","LAD-d",
              "D1","D2","LCx-p","LCx-d","OM1","OM2","Ramus","L-PLB"]
PARENT   = [-1,0,1,2,3,3,0,6,7,8,7,8,6,12,12,13,6,13]
CHILDREN = [[c for c in range(S) if PARENT[c] == p] for p in range(S)]
ANC = np.zeros((S, S), dtype=np.float32)          # ANC[i,k]=1 if k is ancestor-or-self of i
for i in range(S):
    j = i
    while j >= 0: ANC[i, j] = 1; j = PARENT[j]
ADJ = np.eye(S, dtype=np.float32)
for c in range(1, S): ADJ[c, PARENT[c]] = ADJ[PARENT[c], c] = 1
LIN = np.clip(ANC + ANC.T, 0, 1)                   # lineage (ancestor/descendant) mask

CLIN = ["age","sex","bmi","sbp","dbp","htn","bpmed","tchol","hdl","tg","ldl",
        "diabetes","insulin","dm_pills","crp","smoker","sedentary"]
DOMAINS = [[0,1,2],[3,4,5,6],[7,8,9,10],[11,12,13],[14,15,16]]   # demographics, BP, lipids, glycemic, inflammation
P_CLIN = len(CLIN)

## 1. NHANES 2017-2018 + linked mortality (Eq. 6) and IBR-CEI clinical branch (Eq. 8)
Expected layout for real data: `data/nhanes/{DEMO_J,BMX_J,BPX_J,BPQ_J,TCHOL_J,HDL_J,TRIGLY_J,DIQ_J,HSCRP_J,SMQ_J,PAQ_J}.xpt` and `data/nhanes/NHANES_2017_2018_MORT_2019_PUBLIC.dat` (fixed-width NCHS layout).
MICE, robust scaling and the risk atlas are fitted on the training split only (leakage-free).

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

def _rc(s):   # NHANES yes/no recode: 1->1, 2->0, other->nan
    return s.map({1: 1.0, 2: 0.0})

def load_nhanes_real(root, cardiovascular_only=True):
    tabs = ["DEMO_J","BMX_J","BPX_J","BPQ_J","TCHOL_J","HDL_J","TRIGLY_J","DIQ_J","HSCRP_J","SMQ_J","PAQ_J"]
    dfs = []
    for t in tabs:
        p = [q for q in glob.glob(os.path.join(root, "*")) if os.path.basename(q).upper() == t + ".XPT"]
        if not p: raise FileNotFoundError(t)
        dfs.append(pd.read_sas(p[0], format="xport").set_index("SEQN"))
    d = dfs[0].join(dfs[1:], how="left")
    out = pd.DataFrame(index=d.index)
    out["age"] = d["RIDAGEYR"]; out["sex"] = (d["RIAGENDR"] == 1).astype(float); out["bmi"] = d["BMXBMI"]
    out["sbp"] = d[[f"BPXSY{i}" for i in range(1, 5)]].mean(1); out["dbp"] = d[[f"BPXDI{i}" for i in range(1, 5)]].mean(1)
    out["htn"] = _rc(d["BPQ020"]); out["bpmed"] = _rc(d["BPQ040A"])
    out["tchol"] = d["LBXTC"]; out["hdl"] = d["LBDHDD"]; out["tg"] = d["LBXTR"]; out["ldl"] = d["LBDLDL"]
    out["diabetes"] = d["DIQ010"].map({1: 1.0, 2: 0.0, 3: 0.5}); out["insulin"] = _rc(d["DIQ050"]); out["dm_pills"] = _rc(d["DIQ070"])
    out["crp"] = d["LBXHSCRP"]; out["smoker"] = _rc(d["SMQ020"]); out["sedentary"] = d["PAD680"].where(d["PAD680"] < 9000)
    out["w"] = d["WTMEC2YR"]
    mp = glob.glob(os.path.join(root, "*MORT*PUBLIC*.dat"))[0]
    cs = [(0,14),(14,15),(15,16),(16,19),(19,20),(20,21),(42,45),(45,48)]
    m = pd.read_fwf(mp, colspecs=cs, header=None, na_values=["."],
                    names=["SEQN","ELIGSTAT","MORTSTAT","UCOD","DIABETES","HYPERTEN","PERMTH_INT","PERMTH_EXM"]).set_index("SEQN")
    out = out.join(m, how="inner")
    out = out[(out.ELIGSTAT == 1) & (out.age >= 40) & (out.w > 0)]
    out["T"] = out["PERMTH_EXM"].fillna(out["PERMTH_INT"])
    out["E"] = ((out.MORTSTAT == 1) & (out.UCOD.isin([1, 5]) if cardiovascular_only else True)).astype(float)
    out = out.dropna(subset=["T"])
    return out[CLIN + ["w", "T", "E"]].reset_index(drop=True)

def make_nhanes_synth(n, rng):
    age = rng.uniform(40, 85, n); sex = rng.integers(0, 2, n).astype(float); bmi = rng.normal(29, 6, n).clip(16, 55)
    sbp = 105 + 0.5 * (age - 40) + 0.3 * (bmi - 29) + rng.normal(0, 14, n); dbp = 70 + 0.1 * (age - 40) + rng.normal(0, 9, n)
    htn = ((sbp + rng.normal(0, 6, n)) > 135).astype(float); bpmed = htn * (rng.random(n) < 0.7)
    tchol = rng.normal(190, 38, n); hdl = rng.normal(52, 14, n) - 6 * sex; tg = np.exp(rng.normal(4.8, 0.5, n)); ldl = tchol - hdl - tg / 5
    dm = (rng.random(n) < 0.08 + 0.004 * (age - 40) + 0.004 * (bmi - 25).clip(0)).astype(float)
    ins = dm * (rng.random(n) < 0.25); pills = dm * (rng.random(n) < 0.6)
    crp = np.exp(rng.normal(0.6 + 0.03 * (bmi - 29), 0.9, n)); smk = (rng.random(n) < 0.4).astype(float); sed = rng.normal(380, 120, n).clip(30, 900)
    lin = 0.06*(age-60) + 0.012*(sbp-125) + 0.5*dm + 0.35*smk + 0.15*np.log(crp) + 0.004*(ldl-110) - 0.01*(hdl-50) + 0.3*sex
    Tev = -np.log(rng.random(n)) / (0.0045 * np.exp(lin)); C = rng.uniform(24, 60, n)
    X = np.stack([age,sex,bmi,sbp,dbp,htn,bpmed,tchol,hdl,tg,ldl,dm,ins,pills,crp,smk,sed], 1)
    X[rng.random(X.shape) < 0.04] = np.nan
    df = pd.DataFrame(X, columns=CLIN); df["w"] = np.exp(rng.normal(0, 0.5, n)); df["T"] = np.minimum(Tev, C); df["E"] = (Tev <= C).astype(float)
    return df

def prep_nhanes(df, rng, k_bins, min_bin):
    n = len(df); idx = rng.permutation(n)
    # stratified 60/20/20 split on the event indicator
    split = np.zeros(n, dtype=int)
    for e in (0, 1):
        ii = idx[df["E"].values[idx] == e]; a, b = int(.6 * len(ii)), int(.8 * len(ii))
        split[ii[a:b]] = 1; split[ii[b:]] = 2
    X = df[CLIN].values.astype(float); tr = split == 0
    imp = IterativeImputer(max_iter=5, random_state=0, sample_posterior=False).fit(X[tr])
    Xi = imp.transform(X); sc = RobustScaler().fit(Xi[tr]); Xs = sc.transform(Xi).clip(-5, 5).astype(np.float32)
    w = df["w"].values / df["w"].values[tr].mean()
    T = df["T"].values.astype(np.float32); E = df["E"].values.astype(np.float32)
    # risk score (train only) -> weighted ECDF rank
    lr = LogisticRegression(C=1.0, max_iter=500).fit(Xs[tr], E[tr].astype(int), sample_weight=w[tr]) if E[tr].sum() > 5 else None
    score = Xs @ (lr.coef_[0] if lr is not None else np.ones(P_CLIN) / P_CLIN)
    o = np.argsort(score[tr]); s_tr = score[tr][o]; cw = np.cumsum(w[tr][o]); cw = (cw - 0.5 * w[tr][o]) / cw[-1]
    rho = np.interp(score, s_tr, cw).astype(np.float32)
    # equal-mass atlas bins on the training split
    K = k_bins
    while True:
        edges = np.linspace(0, 1, K + 1); bid = np.clip(np.digitize(rho[tr], edges[1:-1]), 0, K - 1)
        if np.bincount(bid, minlength=K).min() >= min_bin or K <= 4: break
        K -= 1
    xt, wt, rt = Xs[tr], w[tr], rho[tr]
    mass = np.array([wt[bid == k].sum() for k in range(K)]); mass /= mass.sum()
    atlas = dict(K=K, mass=mass.astype(np.float32),
                 rho=np.array([np.average(rt[bid == k], weights=wt[bid == k]) for k in range(K)], np.float32),
                 mu=np.stack([np.average(xt[bid == k], axis=0, weights=wt[bid == k]) for k in range(K)]).astype(np.float32),
                 var=np.stack([np.average((xt[bid == k] - np.average(xt[bid == k], axis=0, weights=wt[bid == k]))**2, axis=0, weights=wt[bid == k]) + 1e-3
                               for k in range(K)]).astype(np.float32))
    return dict(x=Xs, w=w.astype(np.float32), T=T, E=E, rho=rho, split=split), atlas

## 2. Canonical coronary graphs
* **Real mode** (`volume_to_stations`): IBR-CEI (B-spline isotropic resampling + HU window, Eq. 7) -> L3DU-MAT (four-level 3D U-Net + medial-axis thinning, Eqs. 9-10) -> station descriptors `[x,y,z,radius,thickness,burden,curvature,attenuation,segment length]` per SCCT slot (ImageCAS-X supplies the 18-segment labels).
* **Synthetic mode**: Murray-consistent trees (gamma = 2.7), slot-wise stenoses driven by a latent severity, scanner attenuation shift and leaf-slot dropout per site, plus a natural-history progression simulator for forecast targets.

In [ ]:
# ---------- L3DU-MAT: segmentation network + preprocessing + station extraction (real data path) ----------
class Block3D(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.net = nn.Sequential(nn.Conv3d(i, o, 3, padding=1), nn.GroupNorm(min(4, o), o), nn.ReLU(inplace=True),
                              nn.Conv3d(o, o, 3, padding=1), nn.GroupNorm(min(4, o), o), nn.ReLU(inplace=True))
    def forward(s, x): return s.net(x)

class L3DUNet(nn.Module):
    """Four-level lightweight 3D U-Net; 4 logits = background, lumen, wall, plaque (softmax -> Eq. 9)."""
    def __init__(s, widths=(8, 16, 32, 64), n_out=4):
        super().__init__()
        s.enc = nn.ModuleList(); c = 1
        for w in widths: s.enc.append(Block3D(c, w)); c = w
        s.up = nn.ModuleList([nn.ConvTranspose3d(widths[i], widths[i-1], 2, stride=2) for i in range(len(widths)-1, 0, -1)])
        s.dec = nn.ModuleList([Block3D(2*widths[i-1], widths[i-1]) for i in range(len(widths)-1, 0, -1)])
        s.head = nn.Conv3d(widths[0], n_out, 1)
    def forward(s, x):
        skips = []
        for i, e in enumerate(s.enc):
            x = e(x)
            if i < len(s.enc) - 1: skips.append(x); x = F.max_pool3d(x, 2)
        for u, d in zip(s.up, s.dec): x = d(torch.cat([u(x), skips.pop()], 1))
        return s.head(x)

def seg_loss(logits, y):                     # Dice + CE
    ce = F.cross_entropy(logits, y); p = logits.softmax(1); oh = F.one_hot(y, logits.shape[1]).permute(0, 4, 1, 2, 3).float()
    inter = (p * oh).sum((2, 3, 4)); den = p.sum((2, 3, 4)) + oh.sum((2, 3, 4))
    return ce + 1 - ((2 * inter + 1) / (den + 1)).mean()

def ibr_resample(vol, spacing, target=0.5, lo=-100., hi=700.):
    from scipy.ndimage import zoom
    z = [sp / target for sp in spacing]; v = zoom(vol.astype(np.float32), z, order=3)
    v = np.clip(v, lo, hi); return (v - v.mean()) / (v.std() + 1e-6)

def volume_to_stations(prob, seg18, spacing=0.5, max_st=8):
    """prob: (4,D,H,W) softmax map of L3DU-MAT; seg18: (D,H,W) ints 0..18 (SCCT segments).
    Returns list of (slot, station_matrix[n,9]) using medial-axis thinning per segment (Eq. 10)."""
    from skimage.morphology import skeletonize
    from scipy.ndimage import distance_transform_edt
    out = []
    lumen = (prob[1:].sum(0) > 0.5)                       # any vessel class
    for slot in range(1, S):
        m = (seg18 == slot) & lumen
        if m.sum() < 20: continue
        sk = skeletonize(m); pts = np.argwhere(sk)
        if len(pts) < 3: continue
        pc = pts - pts.mean(0); ax = np.linalg.svd(pc, full_matrices=False)[2][0]; pts = pts[np.argsort(pc @ ax)]
        dist = distance_transform_edt(m)
        sel = np.linspace(0, len(pts) - 1, min(max_st, len(pts))).astype(int); pts = pts[sel]
        rows = []
        for i, p in enumerate(pts):
            r = max(dist[tuple(p)], 1.0) * spacing
            rad = int(max(3, round(1.6 * r / spacing)))
            sl = tuple(slice(max(0, c - rad), c + rad + 1) for c in p)
            vessel = prob[1:][(slice(None),) + sl].sum(0).mean() + 1e-6
            burden = float(prob[3][sl].mean() / vessel)       # plaque fraction of the local vessel cross-section
            thick = float(prob[2][sl].mean() / vessel) * r
            prev, nxt = pts[max(i-1, 0)], pts[min(i+1, len(pts)-1)]
            a, b = (p - prev) * spacing, (nxt - p) * spacing
            kappa = float(np.linalg.norm(b / (np.linalg.norm(b) + 1e-6) - a / (np.linalg.norm(a) + 1e-6)))
            ell = float(np.linalg.norm(a)) if i > 0 else float(np.linalg.norm(b))
            rows.append([*(p * spacing), r, thick, burden, kappa, 0.0, ell])
        out.append((slot, np.array(rows, np.float32)))
    return out

# ---------- Synthetic generator ----------
GAMMA_TRUE = 2.7
def gen_patient(rng, z, site):
    r = np.zeros(S)
    for k in range(S - 1, 0, -1):
        r[k] = (sum(r[c]**GAMMA_TRUE for c in CHILDREN[k]))**(1/GAMMA_TRUE) * np.exp(rng.normal(0, .03)) if CHILDREN[k] else rng.uniform(.8, 1.4)
    r *= rng.uniform(.9, 1.1)
    sev = 1 / (1 + np.exp(-(z + rng.normal(0, .3))))
    sten = np.where(rng.random(S) < .15 + .5 * sev, rng.uniform(.2, .8, S) * (.4 + .8 * sev), 0).clip(0, .85)
    leaves = [k for k in range(1, S) if not CHILDREN[k]]
    dropped = {k for k in leaves if rng.random() < site["drop"]}
    end = {0: np.zeros(3)}; rows, slots = [], []
    b0 = np.zeros(S); pres = np.zeros(S); rmin = np.zeros(S)
    for k in range(1, S):
        if k in dropped: end[k] = end[PARENT[k]]; continue
        n = rng.integers(4, CFG["max_st"] + 1); d = rng.normal(size=3); d /= np.linalg.norm(d)
        ell = rng.uniform(2, 5, n); pos = end[PARENT[k]] + np.cumsum(ell[:, None] * (d + .25 * rng.normal(size=(n, 3))), 0)
        end[k] = pos[-1]
        rad = r[k] * np.linspace(1.08, .92, n); j = rng.integers(n); sm = np.zeros(n); sm[j] = sten[k]
        for o, f in ((-1, .4), (1, .4)):
            if 0 <= j + o < n: sm[j + o] = sten[k] * f
        rad = rad * (1 - sm)
        b = np.clip(.08 + .30 * sev + .6 * sm + rng.normal(0, .04, n), 0, 1)
        th = .25 + 1.0 * b + rng.normal(0, .03, n); kap = np.abs(rng.normal(0, .08, n))
        att = (350 + site["hu"] - 60 * b + rng.normal(0, 25, n) - 300) / 200
        rows.append(np.stack([pos[:, 0], pos[:, 1], pos[:, 2], rad, th, b, kap, att, ell], 1)); slots += [k] * n
        b0[k] = b.mean(); pres[k] = 1; rmin[k] = rad.min()
    Xp = np.zeros((NST, CFG["d_st"]), np.float32); sl = np.zeros(NST, np.int64); mk = np.zeros(NST, np.float32)
    R = np.concatenate(rows); Xp[:len(R)] = R; sl[:len(R)] = slots; mk[:len(R)] = 1
    # true hemodynamics for progression simulator
    phi = pres * r**GAMMA_TRUE; Phi = phi.copy()
    for k in range(S - 1, -1, -1):
        for c in CHILDREN[k]: Phi[k] += Phi[c]
    q = Phi / max(Phi[0], 1e-6); shear = q / np.maximum(rmin, .2)**3
    shear = np.log1p(shear / (np.median(shear[pres > 0]) + 1e-6))
    rate = .025 * (.4 + 1 / (1 + np.exp(-(1.2 * z + 2.0 * sten + .4 * shear)))) * pres        # per year
    return Xp, sl, mk, b0, pres, rate, sten

def build_imaging_cohort(rng, n):
    z = rng.normal(size=n); tert = np.digitize(z, np.quantile(z, [1/3, 2/3]))
    C = CFG["n_clients"]; props = rng.dirichlet(np.full(C, CFG["dirichlet"]), size=3)     # class -> client proportions
    site_of = np.array([rng.choice(C, p=props[t]) for t in tert])
    sites = [dict(hu=rng.normal(0, 40), drop=rng.uniform(0, .25), vec=np.array([0, 0, 0, 0], np.float32)) for _ in range(C)]
    for c, st in enumerate(sites):
        st["vec"] = np.array([st["hu"] / 100, rng.uniform(.8, 1.2), rng.choice([100, 120, 140]) / 120, rng.uniform(.8, 1.4)], np.float32)
    D = dict(X=[], slot=[], mask=[], b0=[], pres=[], rate=[], sten=[], site=[], client=site_of, z=z)
    for i in range(n):
        Xp, sl, mk, b0, pres, rate, sten = gen_patient(rng, z[i], sites[site_of[i]])
        for k, v in zip(["X","slot","mask","b0","pres","rate","sten"], [Xp, sl, mk, b0, pres, rate, sten]): D[k].append(v)
        D["site"].append(sites[site_of[i]]["vec"] + rng.normal(0, .02, 4).astype(np.float32))
    for k in ["X","slot","mask","b0","pres","rate","sten","site"]: D[k] = np.stack(D[k])
    # follow-up burden targets at 4 horizons (years)
    t = np.array([.25, .5, .75, 1.]) * CFG["horizon_years"]
    y = D["b0"][:, None, :] + D["rate"][:, None, :] * t[None, :, None] * (1 + rng.normal(0, .05, (n, 4, S)))
    D["y"] = np.clip(y, 0, 1).astype(np.float32)
    D["tree_q"] = (D["pres"][:, 1:].mean(1)).astype(np.float32)       # tree-completeness score
    return D

## 3. Model: MP-CTNet (Eqs. 11-18)
Learnable Murray exponent `gamma in [2,3]`, bottom-up potentials -> prior flow shares, nondimensional Poiseuille shear/pressure priors, direction-aware station convolution, slot attention pooling, lineage attention penalised by tree **resistance distance**, and an **exactly conserving** Murray-anchored softmax flow split (children shares sum to 1 by construction).

In [ ]:
class MPCTNet(nn.Module):
    def __init__(s, c):
        super().__init__()
        H = s.H = c["hidden"]; s.L = c["conv_layers"]; s.nh = c["heads"]; s.bound = 1.0
        s.register_buffer("anc", torch.tensor(ANC)); s.register_buffer("lin", torch.tensor(LIN))
        s.register_buffer("feat_mu", torch.zeros(c["d_st"])); s.register_buffer("feat_sd", torch.ones(c["d_st"]))
        s.a_gamma = nn.Parameter(torch.zeros(())); s.c1 = nn.Parameter(torch.zeros(())); s.c2 = nn.Parameter(torch.zeros(()))
        s.inp = nn.Linear(c["d_st"] + 2, H)
        s.w_self = nn.ModuleList([nn.Linear(H, H) for _ in range(s.L)])
        s.w_up = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(s.L)])
        s.w_dn = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(s.L)])
        s.w_edge = nn.ModuleList([nn.Linear(4, H, bias=False) for _ in range(s.L)])
        s.pool_w = nn.Linear(H, H); s.pool_a = nn.Parameter(torch.randn(H) * .1)
        s.slot_emb = nn.Parameter(torch.randn(S, H) * .02)
        s.q, s.k, s.v, s.o = (nn.Linear(H, H) for _ in range(4)); s.slope = nn.Parameter(torch.full((s.nh,), .5))
        s.ln1, s.ln2 = nn.LayerNorm(H), nn.LayerNorm(H)
        s.ffn = nn.Sequential(nn.Linear(H, 2 * H), nn.GELU(), nn.Linear(2 * H, H)); s.flow_head = nn.Linear(H, 1)

    def stats(s, X, slot, mask):
        M = F.one_hot(slot, S).float() * mask[..., None]; n = M.sum(1); pres = (n > 0).float()
        r = X[..., 3].clamp(min=.2)
        rr = r[:, :, None].expand(-1, -1, S).masked_fill(M < .5, float("inf")); srt = rr.sort(1).values
        idx = ((n - 1).clamp(min=0) * .9).floor().long()[:, None, :]
        r90 = torch.where(pres > 0, srt.gather(1, idx).squeeze(1), torch.zeros_like(n))
        rmin = torch.where(pres > 0, srt[:, 0, :], torch.zeros_like(n))
        b0 = (M * X[..., 5:6]).sum(1) / n.clamp(min=1)
        return M, pres, r, r90, rmin, b0

    def murray(s, r90, pres):
        g = 2. + torch.sigmoid(s.a_gamma)
        phi = pres * r90.clamp(min=1e-3).pow(g); Phi = [None] * S
        for k in range(S - 1, -1, -1):
            t = phi[:, k]
            for c in CHILDREN[k]: t = t + Phi[c]
            Phi[k] = t
        Phi = torch.stack(Phi, 1); return g, Phi, Phi / Phi[:, :1].clamp(min=1e-6)

    def hemo(s, q, Phi, g, X, M, r, r90):
        """Nondimensional Poiseuille wall shear tau ~ q/r^3, pressure drop ~ l q / r^4 plus quadratic stenosis term (Eq. 13/18)."""
        rroot = Phi[:, 0].clamp(min=1e-6).pow(1 / g)[:, None]
        qs = torch.einsum("bns,bs->bn", M, q); rr = (r / rroot).clamp(min=.05)
        r90s = (torch.einsum("bns,bs->bn", M, r90) / rroot).clamp(min=.05); ell = X[..., 8].clamp(min=.1) / rroot
        tau = qs / rr.pow(3)
        dp = F.softplus(s.c1 + .5413) * ell * qs / rr.pow(4) + F.softplus(s.c2 + .5413) * qs**2 * (1 / rr**2 - 1 / r90s**2).clamp(min=0)**2
        dp_slot = torch.einsum("bns,bn->bs", M, dp)
        cum = torch.einsum("ik,bk->bi", s.anc, dp_slot) - .5 * dp_slot
        tau_peak = (tau[:, :, None] * M - 1e9 * (1 - M)).amax(1).clamp(min=0)
        tau_mean = (tau[:, :, None] * M).sum(1) / M.sum(1).clamp(min=1)
        hv = torch.log1p(torch.stack([tau_peak, tau_mean, cum.clamp(min=0)], -1))
        return hv, tau, torch.einsum("bns,bs->bn", M, cum)

    def forward(s, X, slot, mask):
        B = X.shape[0]
        M, pres, r, r90, rmin, b0 = s.stats(X, slot, mask)
        g, Phi, qp = s.murray(r90, pres)
        _, tau_s, cum_s = s.hemo(qp, Phi, g, X, M, r, r90)
        f = (X - s.feat_mu) / s.feat_sd
        h = F.relu(s.inp(torch.cat([f, torch.log1p(tau_s.clamp(min=0))[..., None], torch.log1p(cum_s.clamp(min=0))[..., None]], -1))) * mask[..., None]
        # direction-aware station convolution (upstream / downstream neighbours within the slot chain)
        link = (slot[:, 1:] == slot[:, :-1]).float() * mask[:, 1:] * mask[:, :-1]
        ev = torch.cat([X[:, 1:, 0:3] - X[:, :-1, 0:3], X[:, 1:, 8:9]], -1) / 5.
        for l in range(s.L):
            e = s.w_edge[l](ev); up = F.pad((s.w_up[l](h[:, :-1]) + e) * link[..., None], (0, 0, 1, 0))
            dn = F.pad((s.w_dn[l](h[:, 1:]) + e) * link[..., None], (0, 0, 0, 1))
            h = (h + F.relu(s.w_self[l](h) + up + dn)) * mask[..., None]
        # attention pooling inside each canonical slot (Eq. 15)
        sc = (torch.tanh(s.pool_w(h)) @ s.pool_a)[..., None].expand(-1, -1, S).masked_fill(M < .5, -1e9)
        al = sc.softmax(1) * M; z = torch.einsum("bnk,bnh->bkh", al, h) + s.slot_emb[None] * pres[..., None]
        # lineage attention with resistance-distance bias (Eq. 16)
        w = pres / r90.clamp(min=1e-3).pow(4); w = w / (w.sum(1, keepdim=True) / pres.sum(1, keepdim=True).clamp(min=1)).clamp(min=1e-6)
        wa = torch.einsum("ik,bk->bi", s.anc, w); com = torch.einsum("ik,jk,bk->bij", s.anc, s.anc, w)
        Rm = (wa[:, :, None] + wa[:, None, :] - 2 * com).clamp(min=0)
        amask = s.lin[None] * pres[:, :, None] * pres[:, None, :] + torch.eye(S, device=X.device)[None]
        dh = s.H // s.nh
        qh, kh, vh = (m(z).view(B, S, s.nh, dh).transpose(1, 2) for m in (s.q, s.k, s.v))
        sco = qh @ kh.transpose(-1, -2) / math.sqrt(dh) - F.softplus(s.slope)[None, :, None, None] * Rm[:, None] / (Rm.amax((1, 2), keepdim=True)[:, None] + 1e-6)
        att = sco.masked_fill(amask[:, None] < .5, -1e9).softmax(-1)
        z = s.ln1(z + s.o((att @ vh).transpose(1, 2).reshape(B, S, s.H))); z = s.ln2(z + s.ffn(z)) * pres[..., None]
        # exact flow conservation (Eq. 17): Murray-anchored softmax split among present children
        corr = s.flow_head(z).squeeze(-1); lg = torch.log(qp + 1e-4) + s.bound * torch.tanh(corr / s.bound)
        qpass = [None] * S; qpass[0] = torch.ones_like(qp[:, 0])
        for p in range(S):
            ch = CHILDREN[p]
            if not ch: continue
            lo = torch.stack([lg[:, c] for c in ch], 1).masked_fill(torch.stack([pres[:, c] for c in ch], 1) < .5, -1e4)
            sp = lo.softmax(1)
            for i, c in enumerate(ch): qpass[c] = qpass[p] * sp[:, i]
        q = torch.stack(qpass, 1) * pres
        hv, _, _ = s.hemo(q, Phi, g, X, M, r, r90)
        hv = hv * pres[..., None]
        # physics regulariser: Murray residual  sum_c r_c^g = r_p^g  (+ taper)
        res, cnt = 0., 0.
        for p in range(1, S):
            ch = [c for c in CHILDREN[p]]
            if not ch: continue
            lhs = sum(pres[:, c] * r90[:, c].clamp(min=1e-3).pow(g) for c in ch); rhs = r90[:, p].clamp(min=1e-3).pow(g)
            res = res + pres[:, p] * (lhs - rhs).abs() / rhs; cnt = cnt + pres[:, p]
        phys = (res / cnt.clamp(min=1)).mean()
        sten = (1 - rmin / r90.clamp(min=1e-3)) * pres
        tri = torch.stack([(pres * b0).sum(1) / pres.sum(1).clamp(min=1), sten.amax(1), (hv[..., 0] - 1e9 * (1 - pres)).amax(1).clamp(min=0)], -1)
        return dict(z=z, q=q, hemo=hv, R=Rm, pres=pres, b0=b0, phys=phys, gamma=g, tri=tri, qprior=qp, r90=r90)

## 4. RQ-UOTFNet (Eqs. 19-24)
Severity triplet (burden, stenosis, shear) -> rank `rho` -> **per-patient unbalanced entropic OT** to the survey-weighted risk-quantile atlas (log-domain Sinkhorn with KL-relaxed marginals), retained mass `r` as reliability, posterior interpolated with the atlas prior, clinical context sampled from the atlas, five domain tokens, reliability-gated cross-attention.

In [ ]:
class RQUOT(nn.Module):
    def __init__(s, c, atlas):
        super().__init__()
        H = c["hidden"]; s.c = c; K = atlas["K"]
        for k in ["rho", "mass", "mu", "var"]: s.register_buffer("a_" + k, torch.tensor(atlas[k]))
        s.register_buffer("tri_mu", torch.zeros(3)); s.register_buffer("tri_sd", torch.ones(3)); s.register_buffer("sev_sd", torch.ones(()))
        s.w_sev = nn.Parameter(torch.zeros(3))
        s.tok = nn.ModuleList([nn.Linear(len(d), H) for d in DOMAINS]); s.dom = nn.Parameter(torch.randn(len(DOMAINS), H) * .02)
        s.q, s.k, s.v = nn.Linear(H, H), nn.Linear(H, H), nn.Linear(H, H); s.gate = nn.Linear(2 * H, H); s.ln = nn.LayerNorm(H)
    def rank(s, tri):
        sc = (((tri - s.tri_mu) / s.tri_sd) * s.w_sev.softmax(0)).sum(-1)
        return 0.5 * (1 + torch.erf(sc / (s.sev_sd * math.sqrt(2))))
    @torch.no_grad()
    def transport(s, rho):
        c = s.c; C = c["sens"] * (rho[:, None] - s.a_rho[None])**2; eps = c["uot_eps"]
        fa, fb = c["uot_ta"] / (c["uot_ta"] + eps), c["uot_tb"] / (c["uot_tb"] + eps)
        lK = -C / eps; lm = torch.log(s.a_mass[None]); lv = torch.zeros_like(C)
        for _ in range(c["sink_iters"]):
            lu = fa * (0 - torch.logsumexp(lK + lv, 1, keepdim=True)); lv = fb * (lm - (lK + lu))
        pi = torch.exp(lu + lv + lK); tot = pi.sum(1)
        return pi, tot.clamp(max=1.), pi / (tot[:, None] + 1e-8)
    def forward(s, z, rho, pres, train):
        pi, r, om = s.transport(rho)
        w = r[:, None] * om + (1 - r[:, None]) * s.a_mass[None]
        mu, var = w @ s.a_mu, w @ s.a_var
        ctx = mu + (var.sqrt() * torch.randn_like(mu) * .5 if train else 0.)
        toks = torch.stack([s.tok[d](ctx[:, ix]) + s.dom[d] for d, ix in enumerate(DOMAINS)], 1)
        att = (s.q(z) @ s.k(toks).transpose(1, 2) / math.sqrt(z.shape[-1])).softmax(-1); msg = att @ s.v(toks)
        g = torch.sigmoid(s.gate(torch.cat([z, msg], -1)))
        return s.ln(z + r[:, None, None] * g * msg) * pres[..., None], toks.mean(1), r, ctx

## 5. EC-SGAT (Eqs. 25-26), FiLM hypernetwork (Eq. 27) and PPS-GODENet (Eqs. 31-37)

In [ ]:
class ECSGAT(nn.Module):
    def __init__(s, c):
        super().__init__()
        H = c["hidden"]; s.L = c["gat_layers"]; s.register_buffer("adj", torch.tensor(ADJ))
        s.Wl = nn.ModuleList([nn.Linear(H, H) for _ in range(s.L)]); s.Wr = nn.ModuleList([nn.Linear(H, H) for _ in range(s.L)])
        s.We = nn.ModuleList([nn.Linear(3, H, bias=False) for _ in range(s.L)])
        s.a = nn.ParameterList([nn.Parameter(torch.randn(H) * .1) for _ in range(s.L)]); s.ln = nn.ModuleList([nn.LayerNorm(H) for _ in range(s.L)])
        s.pro = nn.Linear(2 * H, H)
    def forward(s, h, q, R, pres):
        ef = torch.stack([q[:, :, None].expand(-1, -1, S), q[:, None, :].expand(-1, S, -1), R / (R.amax((1, 2), keepdim=True) + 1e-6)], -1)
        adj = s.adj[None] * pres[:, :, None] * pres[:, None, :] + torch.eye(S, device=h.device)[None]
        for l in range(s.L):                                     # GATv2 with edge conditioning
            e = F.leaky_relu(s.Wl[l](h)[:, :, None] + s.Wr[l](h)[:, None] + s.We[l](ef), .2) @ s.a[l]
            al = e.masked_fill(adj < .5, -1e9).softmax(-1)
            h = s.ln[l](h + F.elu(al @ s.Wr[l](h))) * pres[..., None]
        mean = h.sum(1) / pres.sum(1, keepdim=True).clamp(min=1); mx = h.masked_fill(pres[..., None] < .5, -1e9).amax(1)
        return h, s.pro(torch.cat([mean, mx], -1))

class FiLMHyper(nn.Module):
    def __init__(s, c):
        super().__init__(); H = c["hidden"]; s.H = H; s.b = c["film_bound"]
        s.net = nn.Sequential(nn.Linear(c["site_dim"], H), nn.ReLU(), nn.Linear(H, 2 * H)); nn.init.zeros_(s.net[2].weight); nn.init.zeros_(s.net[2].bias)
    def forward(s, h, site):
        gb = s.net(site); return (1 + s.b * torch.tanh(gb[:, :s.H]))[:, None] * h + (s.b * torch.tanh(gb[:, s.H:]))[:, None]

class PPSGODE(nn.Module):
    N_SUM = 7
    def __init__(s, c):
        super().__init__(); H = s.H = c["hidden"]; s.c = c; Hp = c["proto_dim"]
        s.register_buffer("Ahat", torch.tensor(ADJ / ADJ.sum(1, keepdims=True)))
        s.Wt = nn.Linear(H, H, bias=False); s.Wu = nn.Linear(4 + H, H); s.vd = nn.Parameter(torch.zeros(H))
        s.Wm = nn.Sequential(nn.Linear(H, H), nn.ReLU(), nn.Linear(H, 1)); nn.init.constant_(s.Wm[2].bias, -3.)
        s.Wv = nn.Linear(H, 1); s.Ws = nn.Linear(H, 4)
        s.pre = nn.Sequential(nn.Linear(s.N_SUM + H, Hp), nn.ReLU(), nn.Linear(Hp, Hp))
        s.proto = nn.Parameter(torch.randn(c["n_proto"], Hp)); u = c["mono_units"]
        s.mw = nn.Parameter(torch.randn(u, s.N_SUM) * .1); s.mb = nn.Parameter(torch.zeros(u)); s.mv = nn.Parameter(torch.randn(u) * .1)
        s.lin_x = nn.Linear(P_CLIN, 1, bias=False); s.lin_d = nn.Linear(Hp, 1, bias=False); s.use_clin = True
    def damp(s): return s.c["d_floor"] + F.softplus(s.vd)
    def field(s, h, u, pres): return (-s.damp() * h + torch.tanh(s.Ahat @ s.Wt(h) + u)) * pres[..., None]
    def integrate(s, h0, u, pres):
        n = s.c["ode_steps"]; dt = 1. / n; h = h0; traj = []
        for i in range(n):                                       # RK4
            k1 = s.field(h, u, pres); k2 = s.field(h + .5*dt*k1, u, pres); k3 = s.field(h + .5*dt*k2, u, pres); k4 = s.field(h + dt*k3, u, pres)
            h = h + dt / 6 * (k1 + 2*k2 + 2*k3 + k4)
            if (i + 1) % (n // 4) == 0: traj.append(h)
        return torch.stack(traj, 1)                              # B,4,S,H
    def mono(s, v): return (F.softplus(s.mv) * F.softplus(F.linear(v, F.softplus(s.mw), s.mb))).sum(-1)
    def forward(s, h0, g, o, tokm, xc):
        pres = o["pres"]; B = h0.shape[0]
        u = s.Wu(torch.cat([o["hemo"], o["q"][..., None], tokm[:, None].expand(-1, S, -1)], -1))
        traj = s.integrate(h0, u, pres)
        mu = o["b0"][:, None] + .2 * F.softplus(s.Wm(traj).squeeze(-1))           # B,4,S
        var = 1e-3 + F.softplus(s.Wv(traj).squeeze(-1)) * .01
        stage = s.Ws(traj[:, -1])
        rate = (mu[:, -1] - o["b0"]) / s.c["horizon_years"]
        n = pres.sum(1).clamp(min=1); neg = -1e9 * (1 - pres); t = 5.
        rmean = (rate * pres).sum(1) / n; rmax = (rate + neg).amax(1).clamp(min=0)
        lse = (torch.logsumexp(t * rate + neg, 1) - torch.log(n)) / t
        rflow = (rate * o["q"]).sum(1) / o["q"].sum(1).clamp(min=1e-6)
        rstd = (((rate - rmean[:, None])**2 * pres).sum(1) / n).sqrt()
        bT = (mu[:, -1] * pres).sum(1) / n; vm = (var[:, -1] * pres).sum(1) / n
        summ = torch.stack([10*rmean, 10*rmax, 10*lse, 10*rflow, 10*rstd, bT, 5*vm.sqrt()], -1)
        e = s.pre(torch.cat([summ, g], -1)); d2 = ((e[:, None] - s.proto[None])**2).sum(-1)
        al = (-d2 / s.c["proto_temp"]).softmax(-1); delta = e - al @ s.proto
        eta = s.mono(summ) + s.lin_d(delta).squeeze(-1) + (s.lin_x(xc).squeeze(-1) if s.use_clin else 0.)
        return dict(h0=h0, eta=eta, mu=mu, var=var, stage=stage, rate=rate, summ=summ, proto_loss=(al * d2).sum(1), alpha=al, traj=traj, u=u)

class FMGLightNet(nn.Module):
    def __init__(s, c, atlas):
        super().__init__(); s.mp = MPCTNet(c); s.rq = RQUOT(c, atlas); s.gat = ECSGAT(c); s.film = FiLMHyper(c); s.ode = PPSGODE(c); s.use_film = True
    def forward(s, b, train=False):
        o = s.mp(b["X"], b["slot"], b["mask"]); rho = s.rq.rank(o["tri"])
        z, tokm, r, ctx = s.rq(o["z"], rho, o["pres"], train)
        h, g = s.gat(z, o["q"], o["R"], o["pres"])
        if s.use_film: h = s.film(h, b["site"])
        out = s.ode(h, g, o, tokm, b["xc"]); out.update(phys=o["phys"], r=r, rho=rho, q=o["q"], pres=o["pres"], b0=o["b0"], gamma=o["gamma"], qprior=o["qprior"])
        return out
    @torch.no_grad()
    def calibrate(s, d):
        X, sl, mk = d["X"], d["slot"], d["mask"]; m = mk[..., None]
        mu = (X * m).sum((0, 1)) / m.sum(); sd = (((X - mu)**2 * m).sum((0, 1)) / m.sum()).sqrt().clamp(min=1e-3)
        s.mp.feat_mu.copy_(mu); s.mp.feat_sd.copy_(sd)
        tri = torch.cat([s.mp(X[i:i+64], sl[i:i+64], mk[i:i+64])["tri"] for i in range(0, len(X), 64)])
        s.rq.tri_mu.copy_(tri.mean(0)); s.rq.tri_sd.copy_(tri.std(0).clamp(min=1e-4))
        sc = ((tri - tri.mean(0)) / tri.std(0).clamp(min=1e-4)).mean(1); s.rq.sev_sd.copy_(sc.std().clamp(min=1e-3))

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 6. Losses (Eqs. 33 and 36)
Cox/ranking terms use a **stop-gradient risk set** (other patients' log-hazards are detached). This makes every per-sample loss depend on one patient's gradient only, which DP-SGD needs; the same code path is used without DP.

In [ ]:
def cox_loss(eta, T, E, re, rT, same):
    risk = (rT[None, :] >= T[:, None]) & ~same
    lg = torch.cat([eta[:, None], re[None, :].expand(len(eta), -1).masked_fill(~risk, -1e9)], 1)
    return (E * (torch.logsumexp(lg, 1) - eta)).mean()

def rank_loss(eta, T, E, re, rT, rE, same, margin, temp):
    p1 = (E[:, None] > 0) & (T[:, None] < rT[None, :]) & ~same
    p2 = (rE[None, :] > 0) & (rT[None, :] < T[:, None]) & ~same
    l1 = F.softplus((margin - (eta[:, None] - re[None, :])) / temp) * temp
    l2 = F.softplus((margin - (re[None, :] - eta[:, None])) / temp) * temp
    return (((l1 * p1).sum(1) + (l2 * p2).sum(1)) / (p1.sum(1) + p2.sum(1)).clamp(min=1)).mean()

def total_loss(out, b, ref, c, parts=False):
    re, rT, rE, same = ref
    pres = out["pres"][:, None, :]; y = b["y"]
    nll = (.5 * (torch.log(out["var"]) + (y - out["mu"])**2 / out["var"]) * pres).sum() / (pres.sum() * 4).clamp(min=1)
    mu_full = torch.cat([out["b0"][:, None], out["mu"]], 1); nr = (F.relu(mu_full[:, :-1] - mu_full[:, 1:]) * pres).sum() / (pres.sum() * 4).clamp(min=1)
    st = torch.bucketize(y[:, -1], torch.tensor([.2, .4, .6], device=y.device))
    ce = (F.cross_entropy(out["stage"].reshape(-1, 4), st.reshape(-1), reduction="none") * out["pres"].reshape(-1)).sum() / out["pres"].sum().clamp(min=1)
    L_prog = nll + c["w_nr"] * nr + c["w_stage"] * ce
    cx = cox_loss(out["eta"], b["T"], b["E"], re, rT, same)
    rk = rank_loss(out["eta"], b["T"], b["E"], re, rT, rE, same, c["rank_margin"], c["rank_temp"])
    L_surv = cx + c["w_rank"] * rk + c["w_proto"] * out["proto_loss"].mean()
    L = c["w_prog"] * L_prog + c["w_surv"] * L_surv + c["w_phys"] * out["phys"]
    return (L, dict(nll=nll.item(), nr=nr.item(), cox=cx.item(), rank=rk.item())) if parts else L

## 7. Cohort assembly (leakage-free), atlas matching and tensors

In [ ]:
def concordance(eta, T, E):
    eta, T, E = map(np.asarray, (eta, T, E)); num = den = 0.
    for i in np.where(E > 0)[0]:
        m = T > T[i]; den += m.sum(); num += (eta[i] > eta[m]).sum() + .5 * (eta[i] == eta[m]).sum()
    return num / max(den, 1)

def td_auc(eta, T, E, t):
    cases = (T <= t) & (E > 0); ctrl = T > t
    if cases.sum() == 0 or ctrl.sum() == 0: return float("nan")
    a, b = eta[cases], eta[ctrl]; return float(((a[:, None] > b[None]).sum() + .5 * (a[:, None] == b[None]).sum()) / (len(a) * len(b)))

def boot_ci(fn, n, B=200, seed=0):
    r = np.random.default_rng(seed); v = [fn(r.integers(0, n, n)) for _ in range(B)]; return np.nanpercentile(v, [2.5, 97.5])

def simulate_sites(D, rng, rho_raw):
    """Real ImageCAS-X has no site metadata in this notebook: create non-IID clients (Dirichlet over severity tertiles),
    scanner attenuation shift and leaf-slot dropout, exactly like the synthetic generator."""
    n = len(D["X"]); C = CFG["n_clients"]; tert = np.digitize(rho_raw, [1/3, 2/3])
    props = rng.dirichlet(np.full(C, CFG["dirichlet"]), size=3); D["client"] = np.array([rng.choice(C, p=props[t]) for t in tert])
    sites = []
    for c in range(C):
        hu = rng.normal(0, 40); sites.append(dict(hu=hu, drop=rng.uniform(0, .25), vec=np.array([hu/100, rng.uniform(.8, 1.2), rng.choice([100, 120, 140])/120, rng.uniform(.8, 1.4)], np.float32)))
    leaf = np.array([k for k in range(1, S) if not CHILDREN[k]])
    for i in range(n):
        st = sites[D["client"][i]]; D["X"][i, :, 7] += (st["hu"] / 200) * D["mask"][i]
        for k in leaf:
            if rng.random() < st["drop"]: D["mask"][i][D["slot"][i] == k] = 0
    D["site"] = np.stack([sites[c]["vec"] + rng.normal(0, .02, 4).astype(np.float32) for c in D["client"]])
    D["tree_q"] = np.array([len(np.unique(D["slot"][i][D["mask"][i] > 0])) / (S - 1) for i in range(n)], np.float32)

def build_all(cfg):
    rng = np.random.default_rng(cfg["seed"]); mode = cfg["mode"]
    if mode == "auto":
        mode = "real" if os.path.isdir(os.path.join(cfg["data_root"], "nhanes")) and os.path.exists(os.path.join(cfg["data_root"], "imagecas_graphs.npz")) else "synthetic"
    print("data mode:", mode)
    if mode == "real":
        nh_df = load_nhanes_real(os.path.join(cfg["data_root"], "nhanes"))
        zz = np.load(os.path.join(cfg["data_root"], "imagecas_graphs.npz")); D = {k: zz[k].copy() for k in ["X", "slot", "mask"]}
        n = len(D["X"]); cfg["n_img"] = n
        if "y" in zz.files: D["y"] = zz["y"].astype(np.float32)
        else: D["y"] = np.zeros((n, 4, S), np.float32); cfg["w_prog"] = 0.; print("no longitudinal targets: progression loss disabled")
    else:
        nh_df = make_nhanes_synth(cfg["n_nhanes"], rng); n = cfg["n_img"]; D = build_imaging_cohort(rng, n)
    nh, atlas = prep_nhanes(nh_df, rng, cfg["n_bins"], cfg["min_bin"])
    sp = np.zeros(n, int); perm = rng.permutation(n); sp[perm[int(.7*n):int(.85*n)]] = 1; sp[perm[int(.85*n):]] = 2; D["split"] = sp
    tmp = MPCTNet(cfg).eval(); t = lambda k, dt=torch.float32: torch.tensor(D[k], dtype=dt)
    with torch.no_grad(): tri = torch.cat([tmp(t("X")[i:i+64], t("slot", torch.long)[i:i+64], t("mask")[i:i+64])["tri"] for i in range(0, n, 64)])
    z = ((tri - tri[sp == 0].mean(0)) / tri[sp == 0].std(0)).mean(1).numpy(); srt = np.sort(z[sp == 0])
    D["rho_raw"] = (np.searchsorted(srt, z) / len(srt)).clip(0, 1).astype(np.float32)
    if mode == "real": simulate_sites(D, rng, D["rho_raw"])
    j = np.zeros(n, int)           # atlas matching: survey-weighted kernel on severity rank, within the same split pool
    for s_ in range(3):
        pool = np.where(nh["split"] == s_)[0]
        for i in np.where(sp == s_)[0]:
            p = nh["w"][pool] * np.exp(-(nh["rho"][pool] - D["rho_raw"][i])**2 / (2 * cfg["match_bw"]**2)); j[i] = rng.choice(pool, p=p / p.sum())
    D["xc"], D["T"], D["E"] = nh["x"][j], nh["T"][j], nh["E"][j]
    print(f"imaging patients {n} | surrogate events {int(D['E'].sum())} ({D['E'].mean():.1%}) | atlas bins {atlas['K']} | NHANES n={len(nh_df)}")
    return D, nh, atlas

def to_tensors(D, idx):
    g = lambda k, dt=torch.float32: torch.tensor(np.asarray(D[k])[idx], dtype=dt, device=DEVICE)
    return dict(X=g("X"), slot=g("slot", torch.long), mask=g("mask"), site=g("site"), xc=g("xc"), T=g("T"), E=g("E"), y=g("y"), q_tree=g("tree_q"))

def take(d, idx):
    idx = torch.as_tensor(idx, device=d["X"].device); return {k: v[idx] for k, v in d.items()}

seed_all(CFG["seed"])
D, NH, ATLAS = build_all(CFG)
parts = {s_: np.where(D["split"] == s_)[0] for s_ in range(3)}
DATA = {nm: to_tensors(D, parts[i]) for i, nm in enumerate(["train", "val", "test"])}

## 8. TQM-FedNet: local DP-SGD, FedProx, tree-quality + update-alignment aggregation (Eqs. 28-30) and RDP accounting (Eq. 39)
* Per-sample gradient clipping to `clip`, Gaussian noise `sigma * clip` per step (local DP-SGD), FedProx proximal term (data-independent post-processing).
* Server weights: `w_c ∝ n_c * tree_quality_c * exp(beta * cos(delta_c, mean delta))`.
* Accountant: small-sampling-rate Gaussian RDP approximation `2 q^2 alpha / sigma^2` per step, converted to (epsilon, delta). It is an **approximation** (not Opacus' exact SGM accountant) and ignores the stop-gradient risk-set dependence, so treat epsilon as indicative.

In [ ]:
def flat(m): return torch.cat([p.detach().reshape(-1) for p in m.parameters() if p.requires_grad])
def set_flat(m, v):
    i = 0
    for p in m.parameters():
        if p.requires_grad: n = p.numel(); p.data.copy_(v[i:i+n].view_as(p)); i += n

def rdp_epsilon(q, sigma, steps, delta):
    return min(steps * 2 * q * q * a / sigma**2 + math.log(1 / delta) / (a - 1) for a in range(2, 257))

def local_update(model, gvec, data, c, fl, gen):
    set_flat(model, gvec); model.train(); model.use_film = fl["film"]; model.ode.use_clin = fl["clin"]
    params = [p for p in model.parameters() if p.requires_grad]; g0 = [p.detach().clone() for p in params]
    opt = torch.optim.Adam(params, lr=c["lr"]); n = len(data["T"]); steps = 0; last = {}
    for ep in range(c["local_epochs"]):
        perm = torch.randperm(n, generator=gen)
        for st in range(0, n, c["batch"]):
            idx = perm[st:st + c["batch"]]; m = len(idx)
            if m < 4: continue
            bt = take(data, idx)
            if fl["dp"]:
                with torch.no_grad(): ref = model(bt, train=False)["eta"]
                acc = [torch.zeros_like(p) for p in params]
                for i in range(m):
                    bi = take(bt, [i]); same = torch.zeros(1, m, dtype=torch.bool, device=DEVICE); same[0, i] = True
                    L = total_loss(model(bi, train=True), bi, (ref, bt["T"], bt["E"], same), c)
                    gr = torch.autograd.grad(L, params, allow_unused=True)
                    nrm = torch.sqrt(sum((g**2).sum() for g in gr if g is not None)); sc = min(1., c["clip"] / (nrm.item() + 1e-8))
                    for a, g in zip(acc, gr):
                        if g is not None: a += g * sc
                for p, a in zip(params, acc): p.grad = (a + c["sigma"] * c["clip"] * torch.randn_like(a)) / m
            else:
                out = model(bt, train=True); same = torch.eye(m, dtype=torch.bool, device=DEVICE)
                L, last = total_loss(out, bt, (out["eta"].detach(), bt["T"], bt["E"], same), c, parts=True)
                opt.zero_grad(); L.backward()
            for p, g in zip(params, g0):
                if p.grad is not None: p.grad += c["mu_prox"] * (p.data - g)
            opt.step(); steps += 1
    return flat(model) - gvec, steps

def aggregate(deltas, ns, taus, c, use_q):
    Dm = torch.stack(deltas); mean = Dm.mean(0); cos = F.cosine_similarity(Dm, mean[None], dim=1)
    w = ns * taus * torch.exp(c["align_beta"] * cos) if use_q else ns.clone()
    w = w / w.sum(); return (w[:, None] * Dm).sum(0), w, cos

## 9. Evaluation utilities: C-index, time-dependent AUC, Breslow baseline hazard / survival curves (Eq. 37), forecast metrics

In [ ]:
@torch.no_grad()
def predict(model, d, bs=64):
    model.eval(); outs = []; n = len(d["T"])
    for i in range(0, n, bs):
        o = model(take(d, torch.arange(i, min(i + bs, n))), train=False); outs.append({k: o[k].cpu() for k in ["eta", "mu", "r", "pres", "rate", "rho", "q"]})
    return {k: torch.cat([o[k] for o in outs]).numpy() for k in outs[0]}

def breslow(eta, T, E):
    ts = np.unique(T[E > 0]); H = []; h = 0.
    for t in ts: h += ((T == t) & (E > 0)).sum() / np.exp(eta[T >= t]).sum(); H.append(h)
    return ts, np.array(H)

def survival_curve(ts, H, eta, grid):
    Ht = np.interp(grid, ts, H, left=0.); return np.exp(-Ht[None] * np.exp(eta[:, None]))

def evaluate(model, name="test", verbose=True, need_train=True):
    pt = predict(model, DATA["train"]) if need_train else None; pv = predict(model, DATA[name]); d = DATA[name]; T, E = d["T"].cpu().numpy(), d["E"].cpu().numpy()
    ci = concordance(pv["eta"], T, E); lo, hi = boot_ci(lambda ix: concordance(pv["eta"][ix], T[ix], E[ix]), len(T), B=100)
    y = d["y"].cpu().numpy()[:, -1]; mu = pv["mu"][:, -1]; m = pv["pres"] > 0
    mae = np.abs(y - mu)[m].mean(); rmse = np.sqrt(((y - mu)[m]**2).mean()); r2 = 1 - ((y - mu)[m]**2).sum() / (((y - y[m].mean())[m]**2).sum() + 1e-9)
    res = dict(cindex=ci, ci_lo=lo, ci_hi=hi, auc12=td_auc(pv["eta"], T, E, 12), auc24=td_auc(pv["eta"], T, E, 24), auc36=td_auc(pv["eta"], T, E, 36),
               burden_MAE=mae, burden_RMSE=rmse, burden_R2=r2, retained_mass=float(pv["r"].mean()))
    if verbose: print(name, {k: round(float(v), 4) for k, v in res.items()})
    return res, pt, pv

def run_federated(c, fl, rounds, seed=0, verbose=True):
    seed_all(seed); gen = torch.Generator().manual_seed(seed)
    model = FMGLightNet(c, ATLAS).to(DEVICE); model.use_film = fl["film"]; model.ode.use_clin = fl["clin"]; model.calibrate(DATA["train"])
    ctr = D["client"][parts[0]]; clients = [np.where(ctr == k)[0] for k in range(c["n_clients"])]
    gvec = flat(model); hist = []; steps_total = 0; wlog = []
    for r in range(rounds):
        deltas, ns, taus, steps_list = [], [], [], []; t0 = time.time()
        for k, idx in enumerate(clients):
            if len(idx) < 8: continue
            cd = take(DATA["train"], torch.as_tensor(idx, device=DEVICE))
            dl, st = local_update(model, gvec, cd, c, fl, gen); deltas.append(dl); steps_list.append(st); ns.append(float(len(idx))); taus.append(float(cd["q_tree"].mean()))
        steps_total += max(steps_list)
        agg, w, cos = aggregate(deltas, torch.tensor(ns, device=DEVICE), torch.tensor(taus, device=DEVICE), c, fl["tq"])
        gvec = gvec + agg; set_flat(model, gvec); wlog.append(w.cpu().numpy())
        res, _, _ = evaluate(model, "val", verbose=False, need_train=False); hist.append(res["cindex"])
        if verbose: print(f"round {r+1:02d}/{rounds} | val C-index {res['cindex']:.4f} | weights {np.round(w.cpu().numpy(), 2)} | {time.time()-t0:.0f}s")
    q = min(1., c["batch"] / min(len(i) for i in clients if len(i) >= 8))      # worst-case client sampling rate
    eps = rdp_epsilon(q, c["sigma"], steps_total, c["delta"]) if fl["dp"] else float("inf")
    return model, hist, np.array(wlog), eps

FULL = dict(film=True, clin=True, dp=True, tq=True)
print("trainable parameters:", count_params(FMGLightNet(CFG, ATLAS)))

## 10. Train the full FMG-LightNet (federated, DP, tree-quality aggregation)

In [ ]:
t0 = time.time()
MODEL, HIST, WLOG, EPS = run_federated(CFG, FULL, CFG["rounds"])
print(f"total training time {time.time()-t0:.0f}s | approx. RDP epsilon (delta={CFG['delta']}): {EPS:.2f}")
RES_TEST, PT, PV = evaluate(MODEL, "test")
os.makedirs("outputs", exist_ok=True); torch.save(MODEL.state_dict(), "outputs/fmg_lightnet.pt")

## 11. Baselines and ablations (same splits, fewer rounds)
* **Clinical-only Cox**: linear Cox on the surrogate NHANES covariates (shows how much the imaging graph adds).
* **w/o tree-quality weighting** (sample-count FedAvg), **w/o DP**, **w/o FiLM**, **image-only** (no clinical linear term).
Re-implementing CorNet / Lahmar / FedStenoNet / DORI needs their original code and is not included; add them as extra rows in `ABL`.

In [ ]:
def clinical_cox(steps=300):
    xt, Tt, Et = DATA["train"]["xc"], DATA["train"]["T"], DATA["train"]["E"]; w = torch.zeros(P_CLIN, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([w], lr=.05, weight_decay=0)
    for _ in range(steps):
        eta = xt @ w; risk = Tt[None] >= Tt[:, None]
        loss = -(Et * (eta - torch.logsumexp(eta[None].expand(len(eta), -1).masked_fill(~risk, -1e9), 1))).mean() + 1e-3 * (w**2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    d = DATA["test"]; return concordance((d["xc"] @ w).detach().cpu().numpy(), d["T"].cpu().numpy(), d["E"].cpu().numpy())

ABL = {"FMG-LightNet (full)": dict(RES_TEST, rounds=CFG["rounds"]), "Clinical-only Cox": dict(cindex=clinical_cox())}
c_abl = dict(CFG)
for nm, fl in {"w/o tree-quality weighting": dict(FULL, tq=False), "w/o DP": dict(FULL, dp=False),
               "w/o FiLM": dict(FULL, film=False), "image-only (no clinical term)": dict(FULL, clin=False)}.items():
    m_, h_, _, e_ = run_federated(c_abl, fl, CFG["ablation_rounds"], verbose=False); r_, _, _ = evaluate(m_, "test", verbose=False, need_train=False); ABL[nm] = dict(r_, eps=e_)
    print(nm, "C-index", round(r_["cindex"], 4))
ABL_DF = pd.DataFrame(ABL).T[["cindex", "auc12", "auc24", "auc36", "burden_MAE", "burden_R2"]].astype(float).round(4); display(ABL_DF) if "display" in globals() else print(ABL_DF)

## 12. Theory checks (Theorems 1 and 2, numerically)

In [ ]:
model = MODEL.eval(); b = take(DATA["test"], torch.arange(min(64, len(DATA["test"]["T"]))))
with torch.no_grad(): o = model(b, train=False)
q, pres = o["q"], o["pres"]
cons = max(((sum(q[:, c] for c in CHILDREN[p]) - q[:, p]).abs() * pres[:, p] * (sum(pres[:, c] for c in CHILDREN[p]) > 0).float()).max().item() for p in range(1, S) if CHILDREN[p])
cons = max(cons, ((q[:, 1] + q[:, 6]) * 1.0 - 1.0).abs().max().item())
print(f"[T1] max flow-conservation residual (parent vs present children): {cons:.2e}   (children shares sum exactly to parent flow)")
dmin = model.ode.damp().min().item(); bound = max(o["h0"].abs().max().item(), 1 / dmin)
print(f"[T1] max |h(t)| = {o['traj'].abs().max().item():.3f} <= max(|h0|, 1/d_min) = {bound:.3f} :", o["traj"].abs().max().item() <= bound + 1e-5)
v = torch.randn(256, PPSGODE.N_SUM, device=DEVICE, requires_grad=True); gmin = torch.autograd.grad(model.ode.mono(v).sum(), v)[0].min().item()
print(f"[T1] min d(log-hazard)/d(progression summary) = {gmin:.3e} (>= 0: partial monotone)")
ts_, H_ = breslow(PT["eta"], DATA["train"]["T"].cpu().numpy(), DATA["train"]["E"].cpu().numpy()); grid = np.linspace(0, 60, 61)
Sv = survival_curve(ts_, H_, PV["eta"], grid); print("[T1] survival curves non-increasing in t:", bool((np.diff(Sv, axis=1) <= 1e-9).all()))
print(f"[T1] attention pairs: station-level {NST**2:,} vs canonical-slot {S**2} ({NST**2 / S**2:.0f}x fewer); lineage-restricted pairs {int(LIN.sum())}")
q_ = CFG["batch"] / max(1, min(len(np.where(D['client'][parts[0]] == k)[0]) for k in range(CFG["n_clients"]) if (D['client'][parts[0]] == k).sum() >= 8))
print(f"[T2] approx. RDP epsilon after training: {EPS:.2f} at delta={CFG['delta']}, sigma={CFG['sigma']}, q~{q_:.2f}")
print("[T2] atlas sensitivity: min bin size", CFG["min_bin"], "-> one subject changes a bin mean by at most O(1/min_bin)")

## 13. Figures

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].plot(range(1, len(HIST) + 1), HIST, marker="o"); ax[0].set(title="Validation C-index per round", xlabel="round", ylabel="C-index")
ax[1].plot(range(1, WLOG.shape[0] + 1), WLOG); ax[1].set(title="Aggregation weights per client", xlabel="round")
T_te, E_te = DATA["test"]["T"].cpu().numpy(), DATA["test"]["E"].cpu().numpy(); tert = np.digitize(PV["eta"], np.quantile(PV["eta"], [1/3, 2/3]))
for g_, nm in enumerate(["low", "mid", "high"]): ax[2].plot(grid, Sv[tert == g_].mean(0), label=f"{nm} risk")
ax[2].set(title="Predicted survival by risk tertile (test)", xlabel="months", ylim=(0, 1.02)); ax[2].legend()
y = DATA["test"]["y"].cpu().numpy()[:, -1]; m = PV["pres"] > 0; ax[3].scatter(y[m], PV["mu"][:, -1][m], s=4, alpha=.4); ax[3].plot([0, 1], [0, 1], "k--")
ax[3].set(title="Plaque burden forecast (slot level)", xlabel="target", ylabel="prediction"); plt.tight_layout(); plt.savefig("outputs/figures.png", dpi=130); plt.show()
gm = float(MODEL.mp.a_gamma.sigmoid().item() + 2); print("learned Murray exponent gamma =", round(gm, 3), "(data-generating gamma = 2.7 in synthetic mode)")

## 14. Edge deployment: latency, INT8 dynamic quantisation, ONNX export
FPS here is measured on **this machine**. The 62.4 FPS Jetson Orin Nano figure must be re-measured on the device (copy `outputs/fmg_lightnet.onnx` and run with ONNX Runtime / TensorRT).

In [ ]:
class EdgeWrap(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, X, slot, mask, site, xc):
        o = s.m(dict(X=X, slot=slot, mask=mask, site=site, xc=xc), train=False); return o["eta"], o["mu"][:, -1]

def bench(fn, n=50):
    for _ in range(5): fn()
    t = time.perf_counter()
    for _ in range(n): fn()
    return n / (time.perf_counter() - t)

cpu_model = copy.deepcopy(MODEL).cpu().eval(); one = take({k: v.cpu() for k, v in DATA["test"].items()}, [0])
args = (one["X"], one["slot"], one["mask"], one["site"], one["xc"]); wrap = EdgeWrap(cpu_model).eval()
with torch.no_grad(): fps32 = bench(lambda: wrap(*args))
print(f"FP32 CPU: {fps32:.1f} patients/s | parameters: {count_params(MODEL):,}")
try:
    qm = torch.ao.quantization.quantize_dynamic(copy.deepcopy(cpu_model), {nn.Linear}, dtype=torch.qint8).eval(); qw = EdgeWrap(qm).eval()
    with torch.no_grad(): fps8 = bench(lambda: qw(*args)); e32, e8 = wrap(*args)[0], qw(*args)[0]
    print(f"INT8 dynamic-quantised CPU: {fps8:.1f} patients/s | |delta log-hazard| = {abs(e32 - e8).item():.4f}")
except Exception as e: print("quantisation skipped:", repr(e)[:150])
try:
    torch.onnx.export(wrap, args, "outputs/fmg_lightnet.onnx", opset_version=17, input_names=["X", "slot", "mask", "site", "xc"], output_names=["log_hazard", "burden_forecast"])
    print("ONNX exported -> outputs/fmg_lightnet.onnx")
except Exception as e: print("ONNX export skipped:", repr(e)[:200])

## 15. Optional real-data path (ImageCAS-X graphs, zero-shot ASOCA Dice)
Run these only when the NIfTI data is available. `preprocess_imagecas` assumes `<case>*img.nii.gz` with a matching `<case>*seg18.nii.gz` (18 SCCT segments); adjust the patterns to the ImageCAS-X release. Attenuation is not extracted by this helper (set to 0); site shift is simulated.

In [ ]:
def train_unet(unet, cases, epochs=50, crop=64, lr=1e-3):
    """cases: list of (normalised_volume[D,H,W], label[D,H,W] in 0..3). Random-crop Dice+CE training."""
    opt = torch.optim.AdamW(unet.parameters(), lr=lr, weight_decay=1e-4); unet.train()
    for ep in range(epochs):
        for v, y in cases:
            z, h, w = [np.random.randint(0, max(1, s_ - crop)) for s_ in v.shape]
            xb = torch.tensor(v[z:z+crop, h:h+crop, w:w+crop][None, None], dtype=torch.float32, device=DEVICE); yb = torch.tensor(y[z:z+crop, h:h+crop, w:w+crop][None], dtype=torch.long, device=DEVICE)
            loss = seg_loss(unet(xb), yb); opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def predict_volume(unet, v, tile=96):
    unet.eval(); pr = np.zeros((4,) + v.shape, np.float32)
    for z in range(0, v.shape[0], tile):
        for h in range(0, v.shape[1], tile):
            for w in range(0, v.shape[2], tile):
                c = v[z:z+tile, h:h+tile, w:w+tile]; pd_ = [(0, (-s_) % 8) for s_ in c.shape]; cp = np.pad(c, pd_)
                o = unet(torch.tensor(cp[None, None], dtype=torch.float32, device=DEVICE)).softmax(1)[0].cpu().numpy()
                pr[(slice(None),) + tuple(slice(a, a + s_) for a, s_ in zip((z, h, w), c.shape))] = o[:, :c.shape[0], :c.shape[1], :c.shape[2]]
    return pr

def preprocess_imagecas(root, unet, out="data/imagecas_graphs.npz", img_pat="img.nii.gz", seg_pat="seg18.nii.gz"):
    import nibabel as nib; from scipy.ndimage import zoom
    Xs, Sl, Mk = [], [], []
    for ip in sorted(glob.glob(os.path.join(root, "**", "*" + img_pat), recursive=True)):
        sp = ip.replace(img_pat, seg_pat)
        if not os.path.exists(sp): continue
        nii = nib.load(ip); zm = nii.header.get_zooms()[:3]; v = ibr_resample(nii.get_fdata(), zm, CFG["voxel_mm"], CFG["hu_lo"], CFG["hu_hi"])
        seg = zoom(nib.load(sp).get_fdata(), [z / CFG["voxel_mm"] for z in zm], order=0).astype(int)
        st = volume_to_stations(predict_volume(unet, v), seg, CFG["voxel_mm"], CFG["max_st"])
        X = np.zeros((NST, CFG["d_st"]), np.float32); sl = np.zeros(NST, np.int64); mk = np.zeros(NST, np.float32); i = 0
        for slot, rows in st: X[i:i+len(rows)] = rows; sl[i:i+len(rows)] = slot; mk[i:i+len(rows)] = 1; i += len(rows)
        Xs.append(X); Sl.append(sl); Mk.append(mk)
    np.savez_compressed(out, X=np.stack(Xs), slot=np.stack(Sl), mask=np.stack(Mk)); print("saved", out, len(Xs), "patients")

def zero_shot_asoca_dice(unet, root):
    import nibabel as nib; ds = []
    for ip in sorted(glob.glob(os.path.join(root, "asoca", "**", "*.nii.gz"), recursive=True)):
        if "label" in ip.lower() or "mask" in ip.lower(): continue
        lp = [p for p in glob.glob(os.path.dirname(ip) + "/*") if "label" in p.lower() or "mask" in p.lower()]
        if not lp: continue
        nii = nib.load(ip); v = ibr_resample(nii.get_fdata(), nii.header.get_zooms()[:3], CFG["voxel_mm"], CFG["hu_lo"], CFG["hu_hi"])
        from scipy.ndimage import zoom
        y = zoom(nib.load(lp[0]).get_fdata(), [z / CFG["voxel_mm"] for z in nii.header.get_zooms()[:3]], order=0) > 0
        pr = predict_volume(unet, v)[1:].sum(0) > .5; ds.append(2 * (pr & y).sum() / (pr.sum() + y.sum() + 1e-6))
    return float(np.mean(ds)) if ds else float("nan")

# Smoke test of the segmentation network on a random patch (always runs):
_u = L3DUNet().to(DEVICE); print("L3DU-MAT params:", count_params(_u), "| output", tuple(_u(torch.randn(1, 1, 32, 32, 32, device=DEVICE)).shape))

## 16. Summary

In [ ]:
summary = dict(parameters=count_params(MODEL), test=RES_TEST, approx_rdp_epsilon=EPS, ablations=json.loads(ABL_DF.to_json()),
               fps_fp32_cpu=fps32)
json.dump(summary, open("outputs/summary.json", "w"), indent=2, default=float); print(json.dumps(summary, indent=2, default=float)[:1500])